# sample_books.json → Qdrant 적재 파이프라인

In [1]:
import sys
sys.path.insert(0, "../..")

import json
from qdrant_client.models import PointStruct

from src.db.qdrant import QdrantDB
from src.embedding.embedder import LocalEmbedder

## 1. 데이터 로드

In [2]:
with open("../../data/raw/sample_books.json", encoding="utf-8") as f:
    books = json.load(f)

print(f"총 {len(books)}권 로드")
print("샘플:", books[0])

총 278권 로드
샘플: {'title': '소년이 온다 :한강 장편소설 ', 'author': '지은이: 한강', 'publisher': '창비', 'publish_year': '2014', 'isbn': '9788936434120', 'genre': '문학 > 한국문학 > 소설', 'loan_count': 3667, 'image_url': 'http://image.aladin.co.kr/product/4086/97/cover/8936434128_2.jpg', 'description': "섬세한 감수성과 치밀한 문장으로 인간 존재의 본질을 탐구해온 작가 한강의 여섯번째 장편소설. '상처의 구조에 대한 투시와 천착의 서사'를 통해 한강만이 풀어낼 수 있는 방식으로 1980년 5월을 새롭게 조명한다."}


## 2. 임베딩 모델 + DB 초기화

In [3]:
embedder = LocalEmbedder()  # BAAI/bge-m3, 1024차원
db = QdrantDB()             # .env에서 QDRANT_URL, QDRANT_API_KEY 읽음

print("컬렉션 목록:", db.get_collections())

/home/jjeong3150/anaconda3/envs/book-curation/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|████████████████████████████████████| 391/391 [00:00<00:00, 11527.92it/s]


컬렉션 목록: collections=[CollectionDescription(name='sample_books'), CollectionDescription(name='books')]


## 3. 컬렉션 생성

In [4]:
COLLECTION = "sample_books"

# 이미 있으면 삭제 후 재생성 (테스트용)
existing = [c.name for c in db.get_collections().collections]
if COLLECTION in existing:
    db.delete_collection(COLLECTION)

db.create_collection(COLLECTION)

🗑️ 컬렉션 삭제 완료: sample_books
✅ 컬렉션 생성 완료: sample_books


## 4. 임베딩 + 적재

In [5]:
def build_embed_text(book):
    # description이 있으면 포함, 없으면 제목+저자+장르만
    parts = [book["title"], book["author"], book["genre"]]
    if book.get("description"):
        parts.append(book["description"])
    return " | ".join(parts)

texts = [build_embed_text(b) for b in books]
print(f"임베딩 시작: {len(texts)}건")
print("예시 텍스트:", texts[0])

vectors = embedder.embed_batch(texts)
print(f"임베딩 완료: 벡터 차원 = {len(vectors[0])}")

임베딩 시작: 278건
예시 텍스트: 소년이 온다 :한강 장편소설  | 지은이: 한강 | 문학 > 한국문학 > 소설 | 섬세한 감수성과 치밀한 문장으로 인간 존재의 본질을 탐구해온 작가 한강의 여섯번째 장편소설. '상처의 구조에 대한 투시와 천착의 서사'를 통해 한강만이 풀어낼 수 있는 방식으로 1980년 5월을 새롭게 조명한다.
임베딩 완료: 벡터 차원 = 1024


In [6]:
points = [
    PointStruct(id="isbn", vector=vec, payload=book)
    for vec, book in zip(vectors, books)
]

# isbn 기준 결정적 UUID → 같은 데이터 재적재 시 upsert로 처리
db.insert(COLLECTION, points, id_field="isbn")

✅ 데이터 적재 완료: 278건 → sample_books


## 5. 검색 테스트

In [7]:
query = "한강 작가의 책"
query_vector = embedder.embed(query)

results = db.search(COLLECTION, query_vector, limit=5, threshold=0.5)

print(f"쿼리: {query}\n")
for r in results:
    print(f"[{r.score:.3f}] {r.payload['title']} / {r.payload['author']}")

쿼리: 한강 작가의 책

[0.638] 소년이 온다 :한강 장편소설  / 지은이: 한강
[0.636] 희랍어 시간 :한강 장편소설  / 지은이: 한강
[0.627] 바람이 분다, 가라 :한강 장편소설  / 한강
[0.620] 채식주의자 :한강 장편소설  / 지은이: 한강
[0.615] 채식주의자:한강 연작소설 / 한강


## 6. 필터 검색 테스트

In [9]:
db.create_payload_index(COLLECTION, "genre", "keyword")

✅ 인덱스 생성 완료: genre (keyword)


In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue, Range

# 예시 1: 장르 일치 (keyword 필터)
f_genre = Filter(must=[
    FieldCondition(key="genre", match=MatchValue(value="문학 > 한국문학 > 소설"))
])

# 예시 2: 출판연도 범위 (integer 필터)
f_year = Filter(must=[
    FieldCondition(key="publish_year", range=Range(gte="2020"))
])

# 예시 3: 장르 + 연도 복합
f_both = Filter(must=[
    FieldCondition(key="genre", match=MatchValue(value="문학 > 한국문학 > 소설")),
    FieldCondition(key="publish_year", range=Range(gte="2020")),
])

# ---- 실행 ----
query = "슬픔과 상실을 다룬 소설"
query_vector = embedder.embed(query)

results = db.search_with_filter(COLLECTION, query_vector, query_filter=f_genre, limit=5, threshold=0.5)

print(f"쿼리: {query}")
print(f"필터: 한국문학 > 소설\n")
for r in results:
    print(f"[{r.score:.3f}] {r.payload['title']} / {r.payload['publish_year']}")